In [ ]:
from langgraph.graph import StateGraph, END

# ==========================================
# 1. DEFINE CONDITIONAL ROUTING LOGIC
# ==========================================
def evaluation_router(state: DischargeSummaryState) -> str:
    """
    Routes the graph based on the System 2 Evaluation.
    If the summary is complete (or max cycles reached), move to Attribution.
    If not, loop back to the Auditor.
    """
    if state.is_summary_complete:
        return "attribution"
    else:
        return "self_evaluation"

# ==========================================
# 2. BUILD THE GRAPH
# ==========================================
def build_discharge_summary_graph():
    # Initialize the graph with our Master State schema
    workflow = StateGraph(DischargeSummaryState)

    # Add all our nodes
    workflow.add_node("extraction", extraction_node)
    workflow.add_node("reconciliation", reconciliation_node)
    workflow.add_node("self_evaluation", self_evaluation_node)
    workflow.add_node("attribution", attribution_node)

    # Define the Edges (The Flow)
    workflow.set_entry_point("extraction")
    workflow.add_edge("extraction", "reconciliation")
    workflow.add_edge("reconciliation", "self_evaluation")
    
    # Conditional Edge for the System 2 Loop
    workflow.add_conditional_edges(
        "self_evaluation",
        evaluation_router,
        {
            "self_evaluation": "self_evaluation", # Loop back to itself to revise
            "attribution": "attribution"          # Move forward when complete
        }
    )
    
    # Finish the graph
    workflow.add_edge("attribution", END)

    # Compile it!
    return workflow.compile()

# ==========================================
# 3. TEST THE FULL COMPILED GRAPH
# ==========================================
# Make sure you have your `parsed_data` from Phase 1 ready
# Let's create a fresh state to run the entire pipeline from scratch

print("--- INITIALIZING GRAPH ---")
app = build_discharge_summary_graph()

initial_state = DischargeSummaryState(
    patient_id=parsed_data["patient_id"],
    preprocessed_age=parsed_data["preprocessed_age"],
    preprocessed_gender=parsed_data["preprocessed_gender"],
    chronological_docs=[SourceDocument(**doc) for doc in parsed_data["chronological_docs"]]
)

print("--- RUNNING FULL PIPELINE ---")
final_state = app.invoke(initial_state)

print("\n--- FINAL GRAPH TRACE ---")
for step in final_state.get("step_execution_trace", []):
    print(f"- {step}")
    
print("\n--- GRAPH COMPILATION SUCCESSFUL ---")